# EnergyScope Exchange Files — Norte Amazónica Bolivia

Generates `Dist.csv` and `Network_exchanges.csv` for the `01_exch/` EnergyScope input folder.

## Methodology
- **Cluster centroids**: geographic mean (arithmetic average of WGS-84 lat/lon) of member municipality coordinates.
- **Pairwise distances**: Haversine formula (geodesic great-circle distance, km). Road distances would be more accurate for exchange cost modelling, but geodesic is sufficient for EnergyScope's linear exchange cost approximation at this scale.
- **Coordinates source**: OpenStreetMap Nominatim / INE Bolivia geographic reference (verified against IGN Bolivia).
- **All `tc_max = 0`**: Norte Amazónica 2025 — all clusters are isolated diesel mini-grids; no existing or planned inter-cluster network connections.

## Re-running
If CLUSTERS or MUNI_COORDS change, re-run all cells. Both CSV files are regenerated automatically via `generate_exch_files()`.

In [11]:
import os
import math
import itertools
import pandas as pd

OUTPUT_DIR = "output_energyscope"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Cluster definitions and municipality coordinates

**CLUSTERS must match `demande.ipynb` exactly** — same keys, same municipality spellings.

In [12]:
CLUSTERS = {
    1: ["Exaltación", "Reyes", "Santa_Rosa_Beni", "Ixiamas"],
    2: ["Bolpebra"],
    3: ["Guayaramerín", "Riberalta", "Puerto_Gonzalo_Moreno"],
    4: ["Bella_Flor", "Filadelfia", "Ingavi", "Nueva_Esperanza", "Porvenir",
        "Puerto_Rico", "San_Lorenzo", "San_Pedro", "Santa_Rosa_Pando",
        "Santos_Mercado", "Sena", "Villa_Nueva"],
    5: ["Cobija"],
}

# WGS-84 coordinates (lat, lon)
# Source: OpenStreetMap Nominatim / INE Bolivia geographic reference
MUNI_COORDS = {
    # Cluster 1 — Beni + Ixiamas (La Paz)
    "Exaltación":            (-13.933, -65.583),
    "Reyes":                 (-14.304, -67.364),
    "Santa_Rosa_Beni":       (-14.067, -66.756),
    "Ixiamas":               (-13.754, -68.129),
    # Cluster 2 — Pando far west
    "Bolpebra":              (-11.098, -68.860),
    # Cluster 3 — Riberalta/Guayaramerín axis
    "Riberalta":             (-10.991, -66.074),
    "Guayaramerín":          (-10.820, -65.397),
    "Puerto_Gonzalo_Moreno": (-11.594, -67.321),
    # Cluster 4 — Central/eastern Pando
    "Bella_Flor":            (-11.030, -67.774),
    "Filadelfia":            (-11.561, -68.673),
    "Ingavi":                (-11.964, -68.161),
    "Nueva_Esperanza":       (-11.508, -67.645),
    "Porvenir":              (-11.250, -68.555),
    "Puerto_Rico":           (-11.107, -67.538),
    "San_Lorenzo":           (-11.480, -67.850),
    "San_Pedro":             (-11.787, -68.364),
    "Santa_Rosa_Pando":      (-10.942, -67.381),
    "Santos_Mercado":        (-11.673, -67.728),
    "Sena":                  (-11.565, -67.172),
    "Villa_Nueva":           (-11.296, -67.717),
    # Cluster 5 — Cobija
    "Cobija":                (-11.022, -68.742),
}

## 2. Helper functions

In [13]:
def centroid(municipalities, coords):
    """Geographic mean of member municipality WGS-84 coordinates."""
    lats = [coords[m][0] for m in municipalities]
    lons = [coords[m][1] for m in municipalities]
    return (sum(lats) / len(lats), sum(lons) / len(lons))


def haversine_km(p1, p2):
    """Great-circle distance in km between two (lat, lon) points (WGS-84)."""
    R = 6371.0  # Earth mean radius, km
    lat1, lon1 = math.radians(p1[0]), math.radians(p1[1])
    lat2, lon2 = math.radians(p2[0]), math.radians(p2[1])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2
    return R * 2 * math.asin(math.sqrt(a))

## 3. Main generation function


## 3a. tc_max assumptions for Network_exchanges.csv

### HVAC_LINE — `tc_max = 0.040 GW` (40 MW) for all pairs

A non-zero `tc_max` tells EnergyScope the optimizer is **allowed** to invest in an
inter-cluster HVAC line; it does not force one. Setting it to 0 would silently
prevent any grid connection regardless of cost.

**Reference:** Pablo's 2050 scenario for Norte Amazónica shows the SA-NO node
receiving a 40 MW interconnection from the national grid. We use this same
40 MW cap uniformly across all C1–C5 pairs as an upper bound on what the model
may build. All clusters are treated as potentially connectable because EnergyScope
routes investments optimally — the model decides whether the connection is
economically justified given distances and demand.

> **Pending validation from supervisor Pablo** on the appropriate `tc_max` value.
> If the cap should be higher (e.g. 100 MW) or pair-specific, update `TC_MAX`
> in the cell below and re-run.

### All other network types — `tc_max = 0`

| Network type | Reason for 0 |
|---|---|
| `HVDC_SUBSEA` | Landlocked region — no submarine cable applicable |
| `GAS_PIPELINE`, `GAS_SUBSEA` | No natural gas infrastructure exists or is planned |
| `DIESEL_PIPELINE`, `GASOLINE_PIPELINE`, `LPG_PIPELINE`, `LFO_PIPELINE`, `JET_FUEL_PIPELINE` | No liquid fuel pipelines; transport is by river/road |
| `H2_RETROFITTED`, `H2_NEW` | H₂ economy not considered for 2025–2050 Norte Amazónica scenario |
| `H2_SUBSEA_RETRO`, `H2_SUBSEA_NEW` | Landlocked region |


In [14]:

# Network types per resource — matches EnergyScope ESMC template
# Order defines the output block order in Network_exchanges.csv
NETWORK_TYPES = [
    ("ELECTRICITY", "HVAC_LINE"),
    ("ELECTRICITY", "HVDC_SUBSEA"),
    ("GAS",         "GAS_PIPELINE"),
    ("GAS",         "GAS_SUBSEA"),
    ("DIESEL",      "DIESEL_PIPELINE"),
    ("GASOLINE",    "GASOLINE_PIPELINE"),
    ("LPG",         "LPG_PIPELINE"),
    ("LFO",         "LFO_PIPELINE"),
    ("JET_FUEL",    "JET_FUEL_PIPELINE"),
    ("H2",          "H2_RETROFITTED"),
    ("H2",          "H2_NEW"),
    ("H2",          "H2_SUBSEA_RETRO"),
    ("H2",          "H2_SUBSEA_NEW"),
]

# tc_max per network type [GW]
# HVAC_LINE = 0.040 GW (40 MW): allows optimizer to invest in connections
# Everything else = 0 (no pipelines exist; subsea types irrelevant — landlocked region)
TC_MAX = {
    "HVAC_LINE": 0.040,
}


def generate_exch_files(clusters_dict, muni_coords, output_dir=OUTPUT_DIR):
    """
    Generate Dist.csv and Network_exchanges.csv from cluster definitions.

    Parameters
    ----------
    clusters_dict : dict  {cluster_id (int): [municipality_name, ...]}
    muni_coords   : dict  {municipality_name: (lat, lon)}
    output_dir    : str   path where both CSVs are saved
    """
    cluster_ids = sorted(clusters_dict.keys())
    labels = {k: f"C{k}" for k in cluster_ids}

    # --- Step 1: compute centroids ---
    centroids = {k: centroid(clusters_dict[k], muni_coords) for k in cluster_ids}

    print("Cluster centroids (lat, lon):")
    for k in cluster_ids:
        lat, lon = centroids[k]
        munis = clusters_dict[k]
        print(f"  C{k}: ({lat:.4f}, {lon:.4f})  [{len(munis)} municipalities: {', '.join(munis)}]")

    # --- Step 2: pairwise distances ---
    pairs = list(itertools.combinations(cluster_ids, 2))
    distances = {}
    for a, b in pairs:
        d = round(haversine_km(centroids[a], centroids[b]), 1)
        distances[(a, b)] = d
        distances[(b, a)] = d  # symmetric

    # --- Step 3: print summary table ---
    print("\nPairwise geodesic distances [km] (Haversine):")
    print(f"  {'Pair':<10} | Distance (km)")
    for a, b in pairs:
        print(f"  C{a} ↔ C{b}   | {distances[(a, b)]:.1f}")

    # --- Step 4: Dist.csv ---
    dist_rows = []
    for a, b in pairs:
        d = distances[(a, b)]
        dist_rows.append({"Region 1": labels[a], "Region 2": labels[b], "dist": d})
        dist_rows.append({"Region 1": labels[b], "Region 2": labels[a], "dist": d})

    df_dist = pd.DataFrame(dist_rows, columns=["Region 1", "Region 2", "dist"])
    dist_path = os.path.join(output_dir, "Dist.csv")
    df_dist.to_csv(dist_path, sep=";", index=False)
    print(f"\nSaved: {dist_path}  ({len(df_dist)} rows)")

    # --- Step 5: Network_exchanges.csv ---
    # Sort order: resource/network_type block → pair order → direction (From→To, To→From)
    # tc_min = 0 for all rows
    # tc_max = 0.040 GW for HVAC_LINE (optimizer may invest); 0 for all others
    exch_rows = []
    for resource, net_type in NETWORK_TYPES:
        tc_max = TC_MAX.get(net_type, 0)
        for a, b in pairs:
            for frm, to in [(a, b), (b, a)]:
                exch_rows.append({
                    "From":         labels[frm],
                    "To":           labels[to],
                    "Resources":    resource,
                    "Network_type": net_type,
                    "tc_min":       0,
                    "tc_max":       tc_max,
                })

    df_exch = pd.DataFrame(exch_rows,
                           columns=["From", "To", "Resources", "Network_type", "tc_min", "tc_max"])
    exch_path = os.path.join(output_dir, "Network_exchanges.csv")
    df_exch.to_csv(exch_path, sep=";", index=False)
    n_pairs = len(pairs)
    n_types = len(NETWORK_TYPES)
    print(f"Saved: {exch_path}  ({len(df_exch)} rows)")
    print(f"  = {n_pairs} pairs × 2 directions × {n_types} network types")

    return df_dist, df_exch


## 4. Run

In [15]:
df_dist, df_exch = generate_exch_files(CLUSTERS, MUNI_COORDS)

print("\n--- Dist.csv preview ---")
print(df_dist.to_string(index=False))

print("\n--- Network_exchanges.csv first 20 rows ---")
print(df_exch.head(20).to_string(index=False))

Cluster centroids (lat, lon):
  C1: (-14.0145, -66.9580)  [4 municipalities: Exaltación, Reyes, Santa_Rosa_Beni, Ixiamas]
  C2: (-11.0980, -68.8600)  [1 municipalities: Bolpebra]
  C3: (-11.1350, -66.2640)  [3 municipalities: Guayaramerín, Riberalta, Puerto_Gonzalo_Moreno]
  C4: (-11.4303, -67.8798)  [12 municipalities: Bella_Flor, Filadelfia, Ingavi, Nueva_Esperanza, Porvenir, Puerto_Rico, San_Lorenzo, San_Pedro, Santa_Rosa_Pando, Santos_Mercado, Sena, Villa_Nueva]
  C5: (-11.0220, -68.7420)  [1 municipalities: Cobija]

Pairwise geodesic distances [km] (Haversine):
  Pair       | Distance (km)
  C1 ↔ C2   | 384.4
  C1 ↔ C3   | 328.9
  C1 ↔ C4   | 304.3
  C1 ↔ C5   | 385.0
  C2 ↔ C3   | 283.3
  C2 ↔ C4   | 113.1
  C2 ↔ C5   | 15.4
  C3 ↔ C4   | 179.2
  C3 ↔ C5   | 270.7
  C4 ↔ C5   | 104.4

Saved: output_energyscope\Dist.csv  (20 rows)
Saved: output_energyscope\Network_exchanges.csv  (260 rows)
  = 10 pairs × 2 directions × 13 network types

--- Dist.csv preview ---
Region 1 Region 2  